# Story C: Behavioral Weirdness — Anomaly Detection

**Module:** Story C | **Focus:** tìm dữ liệu bất thường ở **người dùng** và **phim**  
**Methods used:** **Robust Z-Score / MAD** (ổn định, lâu đời) vs **Isolation Forest** (mới hơn)  
**Author:** Vĩnh Hoàng

## Mục tiêu
Story C không làm segmentation như Story A.  
Thay vào đó, notebook này đi theo hướng **outlier / anomaly detection**:

1. **User anomaly**: người dùng nào có hành vi bất thường?
2. **Movie anomaly**: phim nào có độ phân cực rating cao, hoặc có số lượt rating bất thường?

## Ý tưởng chính
- **Phương pháp cổ điển, ổn định**: Robust Z-Score dựa trên median và MAD.
- **Phương pháp mới hơn**: Isolation Forest.
- So sánh 2 cách bằng:
  - Jaccard overlap của top outliers
  - tương quan score
  - bảng top cases và biểu đồ

## Lưu ý
Notebook giữ nguyên đường dẫn dữ liệu và thư mục output như cũ.

## 0. Imports & Setup

In [15]:
import os
import json
import math
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA


import matplotlib as mpl

# Dark theme for charts so chart text stays readable in notebook / exported HTML
mpl.rcParams.update({
    'figure.facecolor': '#111111',
    'axes.facecolor': '#111111',
    'savefig.facecolor': '#111111',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'axes.edgecolor': 'white',
    'axes.titlecolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'legend.edgecolor': 'white',
    'legend.facecolor': '#111111',
})
sns.set_theme(style='darkgrid', context='notebook')
px.defaults.template = 'plotly_dark'

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR    = 'data-warehousing'
STORY_C_DIR = os.path.join('artifacts', 'story_C')
TABLES_OUT   = os.path.join(STORY_C_DIR, 'tables')
REPORTS_OUT  = os.path.join(STORY_C_DIR, 'reports')
FIGURES_OUT  = os.path.join(STORY_C_DIR, 'figures')

for d in [TABLES_OUT, REPORTS_OUT, FIGURES_OUT]:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42
TOP_PCT = 0.05   # top 5% anomalies
print('✅ Setup OK')

✅ Setup OK


## 1. Load Data

In [16]:
paths = {
    'user_features':  os.path.join(DATA_DIR, 'user_features_train.parquet'),
    'movie_features': os.path.join(DATA_DIR, 'movie_features_train.parquet'),
    'interactions':   os.path.join(DATA_DIR, 'interactions_train.parquet'),
    'movies':         os.path.join(DATA_DIR, 'dim_movies_clean.parquet'),
}

for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing input: {path}')

df_user   = pd.read_parquet(paths['user_features'])
df_movie  = pd.read_parquet(paths['movie_features'])
df_inter  = pd.read_parquet(paths['interactions'])
df_movies = pd.read_parquet(paths['movies'])

print(f'User features : {df_user.shape}')
print(f'Movie features: {df_movie.shape}')
print(f'Interactions  : {df_inter.shape}')
print(f'Movie metadata : {df_movies.shape}')

display(df_user.head(3))

User features : (322397, 29)
Movie features: (76232, 29)
Interactions  : (30296556, 5)
Movie metadata : (86537, 5)


,userId,n_ratings,rating_mean,rating_std,rating_min,rating_max,first_dt,last_dt,active_days,n_tag_events,...,genre_pref__film_noir,genre_pref__horror,genre_pref__imax,genre_pref__musical,genre_pref__mystery,genre_pref__romance,genre_pref__sci_fi,genre_pref__thriller,genre_pref__war,genre_pref__western
0,1,55,3.990909,0.754560,2.0,5.0,2008-11-03 17:31:43+00:00,2008-11-03 18:35:15+00:00,1,0,...,0.0,4.166667,3.000000,3.833333,4.00,4.107143,3.500000,3.750000,4.666667,0.00
1,2,81,3.506173,1.073819,1.0,5.0,1996-06-26 18:59:11+00:00,1996-06-26 19:19:26+00:00,1,0,...,0.0,3.500000,4.333333,3.500000,3.25,3.666667,2.666667,3.428571,4.500000,3.75
2,3,27,4.888889,0.423659,3.0,5.0,2018-09-05 18:57:20+00:00,2018-09-05 19:04:04+00:00,1,0,...,0.0,5.000000,5.000000,0.000000,5.00,5.000000,4.400000,4.875000,5.000000,5.00


## 2. Helper Functions

Các hàm bên dưới dùng để:
- chọn feature số an toàn
- tính Robust Z-score theo MAD
- chuẩn hóa score về cùng thang đo
- ghi file manifest / summary

In [17]:
def safe_numeric_features(df: pd.DataFrame, exclude=None):
    exclude = set(exclude or [])
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols


def robust_zscore_frame(df: pd.DataFrame, eps: float = 1e-9) -> pd.DataFrame:
    """
    Robust z-score using median and MAD:
        z = (x - median) / (1.4826 * MAD + eps)
    """
    med = df.median(axis=0)
    mad = (df - med).abs().median(axis=0)
    denom = 1.4826 * mad.replace(0, np.nan)
    z = (df - med) / (denom + eps)
    return z.replace([np.inf, -np.inf], np.nan).fillna(0)


def minmax01(arr):
    arr = np.asarray(arr, dtype=float)
    mn = np.nanmin(arr)
    mx = np.nanmax(arr)
    return (arr - mn) / (mx - mn + 1e-9)


def top_overlap_stats(set_a, set_b):
    inter = set_a & set_b
    union = set_a | set_b
    jaccard = len(inter) / len(union) if union else 0.0
    return len(inter), len(union), jaccard


def add_if_exists(df, source_df, cols, key='userId'):
    available = [c for c in cols if c in source_df.columns]
    if available and key in source_df.columns and key in df.columns:
        return df.merge(source_df[[key] + available], on=key, how='left')
    return df

print('✅ Helper functions ready')

✅ Helper functions ready


## 3. Preprocessing — User Features

Story C giữ nguyên giá trị thô theo ý nghĩa anomaly signal.  
Không log-transform như Story A, vì extreme values chính là thứ cần phát hiện.

### User anomaly question
- User nào có hành vi rating / activity lệch chuẩn?
- Có thể là user quá “spam”, quá ít hoạt động, hoặc pattern rất khác số đông.

In [18]:
exclude_cols = [c for c in df_user.columns if c in ('userId', 'first_dt', 'last_dt')]

feat_cols = safe_numeric_features(df_user, exclude=exclude_cols)

if not feat_cols:
    raise ValueError('No numeric feature columns found in df_user.')

X_raw = df_user[feat_cols].copy()
X_raw = X_raw.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f'Feature matrix: {X_raw.shape}')
print('Feature columns:')
print(feat_cols)

# Scaled version for Isolation Forest
robust_scaler = RobustScaler()
X_scaled = pd.DataFrame(
    robust_scaler.fit_transform(X_raw),
    columns=feat_cols,
    index=df_user.index
)

# Classical baseline: robust z-score per feature
X_rz = robust_zscore_frame(X_raw)

display(X_raw.head(3))

Feature matrix: (322397, 26)
Feature columns:
['n_ratings', 'rating_mean', 'rating_std', 'rating_min', 'rating_max', 'active_days', 'n_tag_events', 'genre_pref__action', 'genre_pref__adventure', 'genre_pref__animation', 'genre_pref__children', 'genre_pref__comedy', 'genre_pref__crime', 'genre_pref__documentary', 'genre_pref__drama', 'genre_pref__fantasy', 'genre_pref__film_noir', 'genre_pref__horror', 'genre_pref__imax', 'genre_pref__musical', 'genre_pref__mystery', 'genre_pref__romance', 'genre_pref__sci_fi', 'genre_pref__thriller', 'genre_pref__war', 'genre_pref__western']


,n_ratings,rating_mean,rating_std,rating_min,rating_max,active_days,n_tag_events,genre_pref__action,genre_pref__adventure,genre_pref__animation,...,genre_pref__film_noir,genre_pref__horror,genre_pref__imax,genre_pref__musical,genre_pref__mystery,genre_pref__romance,genre_pref__sci_fi,genre_pref__thriller,genre_pref__war,genre_pref__western
0,55,3.990909,0.754560,2.0,5.0,1,0,3.970588,4.090909,4.142857,...,0.0,4.166667,3.000000,3.833333,4.00,4.107143,3.500000,3.750000,4.666667,0.00
1,81,3.506173,1.073819,1.0,5.0,1,0,3.068965,3.444444,3.800000,...,0.0,3.500000,4.333333,3.500000,3.25,3.666667,2.666667,3.428571,4.500000,3.75
2,27,4.888889,0.423659,3.0,5.0,1,0,4.818182,5.000000,5.000000,...,0.0,5.000000,5.000000,0.000000,5.00,5.000000,4.400000,4.875000,5.000000,5.00


## 4. User Anomaly Detection

### Hai phương pháp so sánh
| Method | Loại | Ý tưởng | Tính chất |
|---|---|---|---|
| **Robust Z-Score / MAD** | Cổ điển | Đo khoảng cách tới median theo MAD | Ổn định, dễ giải thích |
| **Isolation Forest** | Mới hơn | Tách điểm bất thường bằng cây ngẫu nhiên | Scale tốt, bắt outlier toàn cục |

### Cách chấm điểm
- **Robust score**: trung bình tuyệt đối của robust z trên các feature.
- **IF score**: `-score_samples()` từ Isolation Forest, càng cao càng bất thường.
- **Combined score**: trung bình sau khi chuẩn hóa 0–1 để tạo ranking cuối.

In [19]:
# ── Robust baseline score ─────────────────────────────────────────────────────
robust_score = X_rz.abs().mean(axis=1)
robust_label = robust_score >= robust_score.quantile(1 - TOP_PCT)

# ── Isolation Forest ──────────────────────────────────────────────────────────
iso = IsolationForest(
    n_estimators=200,
    contamination=TOP_PCT,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iso_label = iso.fit_predict(X_scaled)         # -1 = anomaly, 1 = normal
iso_score = -iso.score_samples(X_scaled)      # higher = more anomalous

# ── Combine & rank ────────────────────────────────────────────────────────────
combined_score = 0.5 * minmax01(robust_score) + 0.5 * minmax01(iso_score)

user_scores = pd.DataFrame({
    'userId': df_user['userId'].values if 'userId' in df_user.columns else np.arange(len(df_user)),
    'robust_score': robust_score.values,
    'robust_label': np.where(robust_label, -1, 1),
    'if_score': iso_score,
    'if_label': iso_label,
    'combined_score': combined_score,
    'method': 'robust_zscore+isolation_forest',
})

# Append a few context columns if they exist
context_cols = [c for c in ['n_ratings', 'active_days', 'n_tag_events', 'n_movies', 'avg_rating', 'std_rating', 'n_genres']
                if c in df_user.columns]
if context_cols:
    user_scores = add_if_exists(user_scores, df_user, context_cols, key='userId')

user_scores['rank'] = user_scores['combined_score'].rank(ascending=False, method='min').astype(int)
user_scores.sort_values('rank', inplace=True)
user_scores.reset_index(drop=True, inplace=True)

print(f'Robust baseline anomalies: {(user_scores["robust_label"] == -1).sum():,}')
print(f'Isolation Forest anomalies: {(user_scores["if_label"] == -1).sum():,}')
print()
display(user_scores[['userId', 'rank', 'combined_score', 'robust_score', 'if_score']].head(10))

Robust baseline anomalies: 16,120
Isolation Forest anomalies: 16,120



,userId,rank,combined_score,robust_score,if_score
0,189614,1,0.716314,37.457839,0.504901
1,48766,2,0.568773,11.358801,0.629243
2,44970,3,0.550654,7.492328,0.649963
3,214831,4,0.550540,9.540047,0.633019
4,161233,5,0.534673,2.735068,0.679337
5,134353,6,0.533192,7.643670,0.637978
6,42604,7,0.502677,2.387473,0.662527
7,209870,8,0.500717,2.339762,0.661715
8,76618,9,0.498430,10.786875,0.590702
9,255448,10,0.493623,3.773557,0.645538


## 5. Movie Anomaly Detection

Story C không chỉ hỏi “user nào weird?” mà còn hỏi:

- **Phim nào gây phân cực?**  
  → `rating_std` cao, tức ý kiến người xem chia mạnh.

- **Phim nào có volume rating bất thường?**  
  → số lượt rating quá ít hoặc quá nhiều so với phần còn lại.

Ở đây notebook dùng cách cổ điển, dễ giải thích:
- **Robust Z-Score / MAD** trên `rating_std`
- nếu cần, thêm một bảng phụ cho `n_ratings` bất thường

Cách này đủ ổn định, không cần method quá “xịn”.

In [20]:
df_movie_base = df_movie.copy()

# Merge metadata if needed
movie_meta_cols = [c for c in ['movieId', 'title', 'genres'] if c in df_movies.columns]
if movie_meta_cols and 'movieId' in df_movie_base.columns:
    df_movie_base = df_movie_base.merge(df_movies[movie_meta_cols].drop_duplicates('movieId'), on='movieId', how='left')

movie_scores = df_movie_base.copy()

# --- Polarization score from rating_std ---
if 'rating_std' in movie_scores.columns:
    med_std = movie_scores['rating_std'].median()
    mad_std = (movie_scores['rating_std'] - med_std).abs().median()
    movie_scores['polarization_score'] = (movie_scores['rating_std'] - med_std) / (1.4826 * mad_std + 1e-9)
    movie_scores['polarization_score'] = movie_scores['polarization_score'].clip(lower=0)
else:
    movie_scores['polarization_score'] = 0.0
    print('⚠️ rating_std not found — polarization_score set to 0')

# --- Volume anomaly from n_ratings (optional) ---
if 'n_ratings' in movie_scores.columns:
    med_n = movie_scores['n_ratings'].median()
    mad_n = (movie_scores['n_ratings'] - med_n).abs().median()
    movie_scores['volume_zscore'] = (movie_scores['n_ratings'] - med_n) / (1.4826 * mad_n + 1e-9)
    movie_scores['volume_anomaly_score'] = movie_scores['volume_zscore'].abs()
else:
    movie_scores['volume_zscore'] = 0.0
    movie_scores['volume_anomaly_score'] = 0.0
    print('⚠️ n_ratings not found — volume anomaly will be skipped')

# Filter out very sparse titles to reduce noise
if 'n_ratings' in movie_scores.columns:
    min_count = max(50, int(movie_scores['n_ratings'].quantile(0.50)))
    movie_scores = movie_scores[movie_scores['n_ratings'] >= min_count].copy()
    print(f'Films with >= {min_count} ratings: {len(movie_scores):,}')

# Final ranking: prioritize polarization
movie_scores['rank_polarization'] = movie_scores['polarization_score'].rank(ascending=False, method='min').astype(int)
movie_scores['rank_volume'] = movie_scores['volume_anomaly_score'].rank(ascending=False, method='min').astype(int)
movie_scores.sort_values(['rank_polarization', 'rank_volume'], inplace=True)
movie_scores.reset_index(drop=True, inplace=True)

cols_to_show = [c for c in ['movieId', 'title', 'genres', 'rating_mean', 'rating_std', 'n_ratings',
                            'polarization_score', 'volume_anomaly_score'] if c in movie_scores.columns]
display(movie_scores[cols_to_show].head(10))

Films with >= 50 ratings: 14,874


,movieId,title,genres,rating_mean,rating_std,n_ratings,polarization_score,volume_anomaly_score
0,148426,Fateful Findings (2013),Drama|Fantasy|Thriller,2.750000,1.821236,72,1.858583,11.297720
1,1311,Santa with Muscles (1996),Comedy,2.433099,1.672927,142,1.568987,23.101309
2,74754,"Room, The (2003)",Comedy|Drama|Romance,2.495150,1.658598,1031,1.541007,173.006880
3,843,Lotto Land (1995),Drama,3.321429,1.638815,56,1.502378,8.599757
4,162660,God's Not Dead 2 (2016),Drama,2.386792,1.628005,53,1.481269,8.093889
5,214398,After We Collided (2020),Romance,2.698276,1.624753,58,1.474920,8.937003
6,110603,God's Not Dead (2014),Drama,2.361538,1.621794,195,1.469141,32.038311
7,59295,Expelled: No Intelligence Allowed (2008),Documentary,2.174497,1.606497,149,1.439272,24.281667
8,133151,Barbie and the Three Musketeers (2009),Animation|Children,3.090909,1.597857,66,1.422402,10.285984
9,256325,The Kissing Booth 3 (2021),Comedy|Romance,2.570000,1.590822,50,1.408664,7.588021


## 6. Visualizations

### 6.1 User Anomaly — Robust Z-Score vs Isolation Forest

In [21]:
# Scatter: robust score vs IF score
plot_users = user_scores.copy()
plot_users['label_if'] = plot_users['if_label'].map({-1: 'Anomaly', 1: 'Normal'})
plot_users['label_robust'] = plot_users['robust_label'].map({-1: 'Anomaly', 1: 'Normal'})

fig = px.scatter(
    plot_users.sample(min(5000, len(plot_users)), random_state=RANDOM_STATE),
    x='robust_score',
    y='if_score',
    color='label_if',
    opacity=0.65,
    title='User Anomaly: Robust Z-Score vs Isolation Forest Score',
    labels={
        'robust_score': 'Robust Z Score (cao = bất thường hơn)',
        'if_score': 'Isolation Forest Score (cao = bất thường hơn)',
        'label_if': 'IF Label'
    },
    color_discrete_map={'Anomaly': '#FF4500', 'Normal': '#7EC8E3'}
)
fig.update_layout(
    width=900,
    height=550,
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    font=dict(family='Inter, sans-serif', size=12, color='white'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.write_html(os.path.join(FIGURES_OUT, 'user_anomaly_scatter.html'))
fig.show()
print('Saved user_anomaly_scatter.html')

Saved user_anomaly_scatter.html


### 6.2 User Anomaly — PCA View

In [22]:
# PCA is only for visualization, not detection
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    'pc1': X_pca[:, 0],
    'pc2': X_pca[:, 1],
    'combined_score': user_scores['combined_score'].values,
    'if_label': user_scores['if_label'].map({-1: 'Anomaly', 1: 'Normal'}).values,
})

fig = px.scatter(
    pca_df.sample(min(6000, len(pca_df)), random_state=RANDOM_STATE),
    x='pc1', y='pc2',
    color='combined_score',
    color_continuous_scale='Viridis',
    opacity=0.65,
    title='User PCA Projection Colored by Combined Anomaly Score',
    labels={'pc1': 'PCA 1', 'pc2': 'PCA 2', 'combined_score': 'Combined score'}
)
fig.update_layout(
    width=900,
    height=550,
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    font=dict(family='Inter, sans-serif', size=12, color='white'),
)
fig.write_html(os.path.join(FIGURES_OUT, 'user_pca_anomaly.html'))
fig.show()
print('Saved user_pca_anomaly.html')

Saved user_pca_anomaly.html


### 6.3 Movie Anomaly — Mean vs Std Dev

In [23]:
if 'rating_mean' in movie_scores.columns and 'rating_std' in movie_scores.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('#111111')
    ax.set_facecolor('#111111')

    scatter = ax.scatter(
        movie_scores['rating_mean'],
        movie_scores['rating_std'],
        c=movie_scores['polarization_score'],
        s=20,
        alpha=0.65
    )
    ax.set_title('Movie Mean Rating vs Std Dev\n(color = polarization score)', color='white')
    ax.set_xlabel('Average Rating', color='white')
    ax.set_ylabel('Rating Standard Deviation', color='white')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('white')

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label('Polarization Score', color='white')
    cbar.ax.tick_params(colors='white')
    plt.setp(cbar.ax.get_yticklabels(), color='white')

    # label top 5
    top5 = movie_scores.head(5)
    for _, row in top5.iterrows():
        title_short = str(row.get('title', row.get('movieId', 'movie')))[:24]
        ax.text(
            row['rating_mean'] + 0.02,
            row['rating_std'] + 0.01,
            title_short,
            fontsize=8,
            fontweight='bold',
            color='white'
        )

    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_OUT, 'movie_mean_vs_std.png'), dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print('Saved movie_mean_vs_std.png')
else:
    print('⚠️ rating_mean or rating_std not available in movie_scores')

Saved movie_mean_vs_std.png


### 6.4 Movie Anomaly — Polarization Histogram

In [24]:
fig = px.histogram(
    movie_scores,
    x='polarization_score',
    nbins=60,
    title='Distribution of Movie Polarization Score (Robust Z-Score)',
    labels={'polarization_score': 'Polarization Score'},
)
for i, (_, row) in enumerate(movie_scores.head(5).iterrows()):
    label = str(row.get('title', row.get('movieId', 'movie')))[:25]
    fig.add_vline(x=row['polarization_score'], line_dash='dash')
    fig.add_annotation(
        x=row['polarization_score'],
        y=0,
        text=label,
        showarrow=True,
        arrowhead=2,
        yshift=10 + i * 24,
        font=dict(size=9, color='white')
    )
fig.update_layout(
    width=900,
    height=480,
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    font=dict(family='Inter, sans-serif', size=12, color='white'),
)
fig.write_html(os.path.join(FIGURES_OUT, 'movie_polarization_hist.html'))
fig.show()
print('Saved movie_polarization_hist.html')

Saved movie_polarization_hist.html


## 7. Evaluation

Không có ground truth nhãn thật cho anomaly, nên đánh giá theo:
- overlap giữa 2 phương pháp
- tương quan score
- sanity check top-k cases

In [25]:
# Top-k sets
k_user = max(1, int(len(user_scores) * TOP_PCT))
top_robust = set(user_scores.nsmallest(k_user, 'rank')['userId'])
top_if = set(user_scores[user_scores['if_label'] == -1]['userId'])

inter_n, union_n, jaccard = top_overlap_stats(top_robust, top_if)
pearson = user_scores['robust_score'].corr(user_scores['if_score'])
spearman = user_scores['robust_score'].corr(user_scores['if_score'], method='spearman')

print('=' * 60)
print(f'User top-{k_user} overlap')
print(f'  Robust flagged : {len(top_robust):,}')
print(f'  IF flagged     : {len(top_if):,}')
print(f'  Intersection   : {inter_n:,}')
print(f'  Union          : {union_n:,}')
print(f'  Jaccard        : {jaccard:.3f}')
print(f'  Pearson r      : {pearson:.3f}')
print(f'  Spearman ρ     : {spearman:.3f}')
print('=' * 60)

if jaccard > 0.30:
    print('→ Strong agreement: both methods find similar anomalies.')
elif jaccard > 0.15:
    print('→ Moderate agreement: the methods overlap, but each still finds unique cases.')
else:
    print('→ Low agreement: the methods are capturing different anomaly patterns.')

User top-16119 overlap
  Robust flagged : 16,119
  IF flagged     : 16,120
  Intersection   : 15,396
  Union          : 16,843
  Jaccard        : 0.914
  Pearson r      : 0.839
  Spearman ρ     : 0.872
→ Strong agreement: both methods find similar anomalies.


In [26]:
# Score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111111')
fig.suptitle('User Anomaly Score Distribution', fontsize=13, fontweight='bold', color='white')

for ax in axes:
    ax.set_facecolor('#111111')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('white')

sns.histplot(user_scores['combined_score'], bins=50, kde=True, ax=axes[0])
axes[0].axvline(user_scores['combined_score'].quantile(1 - TOP_PCT), linestyle='--')
axes[0].set_title('Histogram + KDE', color='white')
axes[0].set_xlabel('Combined Score', color='white')
axes[0].grid(alpha=0.25)

sns.boxplot(x=user_scores['combined_score'], ax=axes[1])
axes[1].set_title('Boxplot', color='white')
axes[1].set_xlabel('Combined Score', color='white')
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_OUT, 'eval_user_score_distribution.png'), dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved eval_user_score_distribution.png')

Saved eval_user_score_distribution.png


In [27]:
# Movie polarization summary
movie_eval = pd.DataFrame({
    'Metric': [
        'Total Users Analyzed',
        'Robust Top Users Count',
        'IF Anomaly Count',
        'User Jaccard (Top-k vs IF)',
        'User Pearson Corr',
        'User Spearman Corr',
        'Total Movies Analyzed',
        'Top Polarizing Movies (Top 5%)',
        'Max Polarization Score',
        'Max Volume Anomaly Score',
    ],
    'Value': [
        int(len(user_scores)),
        int(len(top_robust)),
        int(len(top_if)),
        round(float(jaccard), 4),
        round(float(pearson), 4) if pd.notna(pearson) else None,
        round(float(spearman), 4) if pd.notna(spearman) else None,
        int(len(movie_scores)),
        int(max(1, int(len(movie_scores) * TOP_PCT))),
        round(float(movie_scores['polarization_score'].max()), 4) if 'polarization_score' in movie_scores.columns else None,
        round(float(movie_scores['volume_anomaly_score'].max()), 4) if 'volume_anomaly_score' in movie_scores.columns else None,
    ]
})

display(movie_eval)
movie_eval.to_csv(os.path.join(REPORTS_OUT, 'eval_story_c_metrics.csv'), index=False)
print('Saved eval_story_c_metrics.csv')

,Metric,Value
0,Total Users Analyzed,322397.0000
1,Robust Top Users Count,16119.0000
2,IF Anomaly Count,16120.0000
3,User Jaccard (Top-k vs IF),0.9141
4,User Pearson Corr,0.8388
5,User Spearman Corr,0.8723
6,Total Movies Analyzed,14874.0000
7,Top Polarizing Movies (Top 5%),743.0000
8,Max Polarization Score,1.8586
9,Max Volume Anomaly Score,19520.4371


Saved eval_story_c_metrics.csv


## 8. Export Artifacts

In [28]:
# Tables
user_scores.to_parquet(os.path.join(TABLES_OUT, 'user_anomaly_scores.parquet'), index=False)
movie_scores.to_parquet(os.path.join(TABLES_OUT, 'movie_anomaly_scores.parquet'), index=False)
print('Saved anomaly score tables')

# JSON summary
summary = {
    'story': 'C',
    'timestamp_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'data_dir': DATA_DIR,
    'top_pct': TOP_PCT,
    'user_count': int(len(user_scores)),
    'movie_count': int(len(movie_scores)),
    'user_jaccard': float(jaccard),
    'user_pearson': float(pearson) if pd.notna(pearson) else None,
    'user_spearman': float(spearman) if pd.notna(spearman) else None,
    'methods': ['Robust Z-Score / MAD', 'Isolation Forest'],
    'notes': 'Story C focuses on anomaly detection for both users and movies.'
}
with open(os.path.join(REPORTS_OUT, 'story_c_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Saved story_c_summary.json')

# Case study markdown
lines = []
lines.append('# Story C: Behavioral Weirdness — Case Studies')
lines.append('')
lines.append('Auto-generated summary for top anomalies.')
lines.append('')
lines.append('## Top Anomalous Users')
lines.append('')
for _, row in user_scores.head(8).iterrows():
    uid = row['userId']
    lines.append(f'### User {uid} (Rank #{int(row["rank"])})')
    lines.append(f'- Robust score: {row["robust_score"]:.4f}')
    lines.append(f'- IF score: {row["if_score"]:.4f}')
    lines.append(f'- Combined score: {row["combined_score"]:.4f}')
    for c in ['n_ratings', 'active_days', 'n_tag_events', 'n_movies', 'avg_rating', 'std_rating', 'n_genres']:
        if c in row and pd.notna(row[c]):
            lines.append(f'- {c}: {row[c]}')
    lines.append('')

lines.append('## Top Polarizing Movies')
lines.append('')
lines.append('| Rank | Title | Mean | Std | # Ratings | Polarization Score | Volume Score |')
lines.append('|---|---|---:|---:|---:|---:|---:|')
for _, row in movie_scores.head(10).iterrows():
    title = str(row.get('title', row.get('movieId', 'movie')))[:50].replace('|', ' ')
    mean_r = f"{row['rating_mean']:.2f}" if 'rating_mean' in row and pd.notna(row.get('rating_mean')) else '?'
    std_r = f"{row['rating_std']:.2f}" if 'rating_std' in row and pd.notna(row.get('rating_std')) else '?'
    n_r = f"{int(row['n_ratings'])}" if 'n_ratings' in row and pd.notna(row.get('n_ratings')) else '?'
    pol = f"{row['polarization_score']:.4f}" if 'polarization_score' in row and pd.notna(row.get('polarization_score')) else '?'
    vol = f"{row['volume_anomaly_score']:.4f}" if 'volume_anomaly_score' in row and pd.notna(row.get('volume_anomaly_score')) else '?'
    lines.append(f'| {int(row["rank_polarization"])} | {title} | {mean_r} | {std_r} | {n_r} | {pol} | {vol} |')

case_path = os.path.join(REPORTS_OUT, 'story_c_case_studies.md')
with open(case_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'Saved {case_path}')

Saved anomaly score tables
Saved story_c_summary.json
Saved artifacts\story_C\reports\story_c_case_studies.md


## 9. Kết Luận

### User anomaly
- **Robust Z-Score / MAD** là baseline cổ điển, dễ giải thích, rất phù hợp để bắt điểm lệch mạnh theo từng feature.
- **Isolation Forest** là cách mới hơn, bắt outlier đa chiều tốt hơn khi profile user phức tạp.
- So sánh 2 cách giúp biết anomaly đang là:
  - outlier toàn cục
  - hay điểm lệch theo một số chiều riêng

### Movie anomaly
- `rating_std` cao → phim gây phân cực mạnh.
- `n_ratings` bất thường → phim có volume rating không bình thường.
- Hai góc nhìn này bổ sung nhau, nên Story C có thể dùng để trình bày business case rõ ràng hơn.

### Kết luận thực dụng
Nếu phải chọn một story đơn giản và rõ:
- **User weirdness** = phát hiện người dùng bất thường
- **Movie weirdness** = phát hiện phim phân cực bất thường

Notebook này giữ cả hai để story C đủ mạnh, nhưng vẫn dùng method đơn giản và dễ defend.